# EXPERTO 3 — ISIC 2019 Skin Lesion Classification
 8 clases: MEL, NV, BCC, AK, BKL, DF, VASC, SCC
 (UNK ignorada — 0 muestras)
 Arquitectura: EfficientNet-B3 (SOTA clásico para ISIC)
 Loss: CrossEntropyLoss con class weights
 Métrica objetivo: F1 Macro > 0.72

EXPERTO 3 — ISIC 2019 Skin Lesion Classification
- Arquitectura : ConvNeXt-Small (timm) — diferente a EfficientNet-B3 (Osteoartritis)
- Dataset      : ISIC 2019 — 8 clases dermatologicas (UNK excluida)

Por que ConvNeXt-Small y no EfficientNet-B3:
  - Depthwise conv 7x7 + LayerNorm + GELU  (vs MBConv + BN + Swish)
  - Sin Squeeze-and-Excitation, sin compound scaling
  - Paradigma "ViT-ified ConvNet" vs "mobile CNN escalado"
  - ~50M params vs ~12M
 
Proceso de entrenamiento:
  FASE 1 (epocas  1– 5): Solo head, backbone congelado. Warm-up rapido.
  FASE 2 (epocas  6–20): Fine-tuning diferencial (ultimos 2 stages).
  FASE 3 (epocas 21–40): Fine-tuning global. LRs reducidos. Early stopping.
 
FIXES Windows + Jupyter:
  - num_workers=0  → evita freeze/deadlock en Jupyter Windows
  - persistent_workers eliminado
  - f1 removido del postfix por batch (caro con miles de samples acumulados)
  - GradScaler con sintaxis nueva para evitar FutureWarning

In [1]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler          # sintaxis nueva, sin FutureWarning
 
import timm
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

C:\Users\isape\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.environ["KAGGLE_CONFIG_DIR"] = r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\.kaggle"

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

DOWNLOAD_DIR = r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\isic"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Descargando ISIC 2019 (~4 GB)...")
api.dataset_download_files("andrewmvd/isic-2019",
                           path=DOWNLOAD_DIR,
                           unzip=True)
print("Descarga completa")

Descargando ISIC 2019 (~4 GB)...
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/isic-2019
Descarga completa


In [3]:
# Verificar estructura
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    depth = root.replace(DOWNLOAD_DIR, '').count(os.sep)
    if depth > 2: continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth == 2:
        print(f"{indent}  ({len(files)} archivos)")

isic/
  ISIC_2019_Training_Input/
    ISIC_2019_Training_Input/
      (25333 archivos)


In [4]:
import pandas as pd
from pathlib import Path

isic_root = Path(r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\isic")
df = pd.read_csv(isic_root / "ISIC_2019_Training_GroundTruth.csv")
print(df.shape)
print(df.head(3))
print(df.columns.tolist())

# Distribución de clases
class_cols = [c for c in df.columns if c != "image"]
counts = df[class_cols].sum().sort_values(ascending=False)
print("\nDistribución:")
print(counts)

(25331, 10)
          image  MEL   NV  BCC   AK  BKL   DF  VASC  SCC  UNK
0  ISIC_0000000  0.0  1.0  0.0  0.0  0.0  0.0   0.0  0.0  0.0
1  ISIC_0000001  0.0  1.0  0.0  0.0  0.0  0.0   0.0  0.0  0.0
2  ISIC_0000002  1.0  0.0  0.0  0.0  0.0  0.0   0.0  0.0  0.0
['image', 'MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

Distribución:
NV      12875.0
MEL      4522.0
BCC      3323.0
BKL      2624.0
AK        867.0
SCC       628.0
VASC      253.0
DF        239.0
UNK         0.0
dtype: float64


In [5]:
import pandas as pd
from pathlib import Path

isic_root = Path(r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\isic")
df = pd.read_csv(isic_root / "ISIC_2019_Training_Metadata.csv")
print(df.shape)
print(df.head(3))
print(df.columns.tolist())


(25331, 5)
          image  age_approx anatom_site_general lesion_id     sex
0  ISIC_0000000        55.0      anterior torso       NaN  female
1  ISIC_0000001        30.0      anterior torso       NaN  female
2  ISIC_0000002        60.0     upper extremity       NaN  female
['image', 'age_approx', 'anatom_site_general', 'lesion_id', 'sex']


In [6]:
# ─────────────────────────────────────────────
# 0. REPRODUCIBILIDAD Y GPU
# ─────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
 
device = torch.device("cuda")
assert torch.cuda.is_available(), (
    "\n[ERROR] Este script requiere GPU CUDA.\n"
    "Asegurate de correrlo en el entorno con PyTorch CUDA instalado."
)
 
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print("=" * 60)
print(f"  GPU  : {gpu_name}")
print(f"  VRAM : {vram_gb:.1f} GB")
print("=" * 60)

  GPU  : NVIDIA GeForce RTX 3050 6GB Laptop GPU
  VRAM : 6.4 GB


In [7]:
# ─────────────────────────────────────────────
# 1. CONFIGURACION
# ─────────────────────────────────────────────
ISIC_ROOT  = Path(r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\isic")
IMG_DIR    = ISIC_ROOT / "ISIC_2019_Training_Input" / "ISIC_2019_Training_Input"
GT_CSV     = ISIC_ROOT / "ISIC_2019_Training_GroundTruth.csv"
 
IMG_SIZE     = 224
BATCH_SIZE   = 32          # subido de 24 → mayor throughput con num_workers=0
ACCUM_STEPS  = 3           # batch efectivo = 96 (igual que antes)
NUM_WORKERS  = 0           # CRITICO: 0 evita freeze/deadlock en Windows + Jupyter
NUM_EPOCHS   = 40
NUM_CLASSES  = 8           # UNK excluida (0 muestras en el dataset)
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-5
WEIGHT_DECAY = 1e-4
PATIENCE     = 8
 
CLASS_NAMES  = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
 
CHECKPOINT_PATH = Path("expert3_convnext_isic2019_best.pth")
PLOT_PATH       = Path("expert3_training_curves.png")
 

In [8]:
# ─────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────
def build_dataframe(gt_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(gt_csv)
    df = df[df["UNK"] != 1.0].copy()
    df["label"]    = df[CLASS_NAMES].values.argmax(axis=1)
    df["filepath"] = df["image"].apply(lambda x: str(IMG_DIR / f"{x}.jpg"))
    return df[["image", "filepath", "label"]].reset_index(drop=True)
 
 
class ISICDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        return self.transform(img), int(row["label"])
 

In [9]:
# ─────────────────────────────────────────────
# 3. TRANSFORMS — AUGMENTATION AGRESIVO
# ─────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
 
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])
 
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
 

In [10]:
# ─────────────────────────────────────────────
# 4. CLASS WEIGHTS Y SAMPLER
# ─────────────────────────────────────────────
def compute_class_weights(df: pd.DataFrame) -> torch.Tensor:
    counts  = df["label"].value_counts().sort_index()
    counts  = counts.reindex(range(NUM_CLASSES), fill_value=1).values.astype(float)
    weights = 1.0 / counts
    weights = weights / weights.sum() * NUM_CLASSES
    return torch.tensor(weights, dtype=torch.float32)
 
 
def build_weighted_sampler(df: pd.DataFrame, cw: torch.Tensor):
    sw = cw[df["label"].values]
    return WeightedRandomSampler(weights=sw, num_samples=len(df), replacement=True)


In [11]:
# ─────────────────────────────────────────────
# 5. MODELO — ConvNeXt-Small
# ─────────────────────────────────────────────
class Expert3_ConvNeXtSmall(nn.Module):
    """
    Backbone : ConvNeXt-Small preentrenado ImageNet-1K
    Head     : LayerNorm -> Dropout(0.4) -> Linear(768->256) -> GELU -> Dropout(0.3) -> Linear(256->8)
 
    Diferencia arquitectonica vs EfficientNet-B3 (Osteoartritis):
      Paradigma  | EfficientNet-B3: Mobile CNN escalado    | ConvNeXt-Small: ViT-ified ConvNet
      Norm       | BatchNorm                               | LayerNorm
      Activacion | Swish                                   | GELU
      Modulo     | Squeeze-Excitation                      | Depthwise conv 7x7
      Params     | ~12M                                    | ~50M
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            "convnext_small", pretrained=pretrained,
            num_classes=0, global_pool="avg"
        )
        feat_dim = self.backbone.num_features  # 768 para convnext_small
 
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES),
        )
        nn.init.trunc_normal_(self.head[-1].weight, std=0.02)
        nn.init.zeros_(self.head[-1].bias)
 
    def forward(self, x):
        return self.head(self.backbone(x))
 
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
 
    def unfreeze_backbone(self, last_n_stages: int = 2):
        for stage in list(self.backbone.stages)[-last_n_stages:]:
            for p in stage.parameters():
                p.requires_grad = True
        for p in self.backbone.head.parameters():
            p.requires_grad = True
 
    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True
 

In [12]:
# ─────────────────────────────────────────────
# 6. TRAIN / VAL CON TQDM
# ─────────────────────────────────────────────
def vram_str() -> str:
    used = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    return f"{used:.1f}/{peak:.1f}GB"
 
 
def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, total_epochs):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()
 
    bar = tqdm(
        enumerate(loader), total=len(loader),
        desc=f"  E{epoch:>2}/{total_epochs} [TRAIN]",
        ncols=90, leave=False,
    )
 
    for step, (imgs, labels) in bar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device)
 
        with autocast("cuda"):
            logits = model(imgs)
            loss   = criterion(logits, labels) / ACCUM_STEPS
 
        scaler.scale(loss).backward()
 
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
 
        running_loss += loss.item() * ACCUM_STEPS
        all_preds.extend(logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
 
        # Solo loss y VRAM en el postfix (f1 por batch es muy caro)
        if step % 20 == 0:
            bar.set_postfix(
                loss=f"{running_loss / (step + 1):.4f}",
                VRAM=vram_str(),
            )
 
    avg_loss = running_loss / len(loader)
    final_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, final_f1
 
 
@torch.no_grad()
def validate(model, loader, criterion, epoch, total_epochs):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    bar = tqdm(
        loader, total=len(loader),
        desc=f"  E{epoch:>2}/{total_epochs} [VAL  ]",
        ncols=90, leave=False,
    )
 
    for imgs, labels in bar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device)
        with autocast("cuda"):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        bar.set_postfix(
            loss=f"{total_loss / (bar.n + 1):.4f}",
            VRAM=vram_str(),
        )
 
    avg_loss = total_loss / len(loader)
    f1_val   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, f1_val, all_preds, all_labels
 

In [13]:
# ─────────────────────────────────────────────
# 7. TABLA RESUMEN POR EPOCA
# ─────────────────────────────────────────────
HEADER = (
    f"{'Epoca':>6} | {'Fase':<7} | "
    f"{'L_train':>8} | {'F1_train':>9} | "
    f"{'L_val':>8} | {'F1_val':>8} | "
    f"{'VRAM_peak':>10} | {'Best':>4}"
)
SEP = "-" * len(HEADER)
 
 
def print_epoch_row(epoch, phase, lt, ft, lv, fv, is_best):
    peak = torch.cuda.max_memory_allocated() / 1e9
    mark = " <<" if is_best else ""
    print(
        f"{epoch:>6} | {phase:<7} | "
        f"{lt:>8.4f} | {ft:>9.4f} | "
        f"{lv:>8.4f} | {fv:>8.4f} | "
        f"{peak:>9.2f}G |{mark}"
    )
 

In [14]:
# ─────────────────────────────────────────────
# 8. GRAFICA DE CURVAS
# ─────────────────────────────────────────────
def plot_curves(history: dict, save_path: Path):
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor("#0f172a")
 
    for ax in axes:
        ax.set_facecolor("#1e293b")
        ax.tick_params(colors="#94a3b8")
        ax.xaxis.label.set_color("#94a3b8")
        ax.yaxis.label.set_color("#94a3b8")
        ax.title.set_color("#f1f5f9")
        for spine in ax.spines.values():
            spine.set_edgecolor("#334155")
 
    # Loss
    axes[0].plot(epochs, history["train_loss"], color="#3b82f6", lw=2, label="Train Loss")
    axes[0].plot(epochs, history["val_loss"],   color="#f59e0b", lw=2, label="Val Loss", linestyle="--")
    axes[0].set_title("Loss por Epoca — Expert 3 ISIC 2019")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("CrossEntropy Loss")
    axes[0].legend(facecolor="#334155", labelcolor="#f1f5f9")
    axes[0].grid(alpha=0.2, color="#475569")
    for ep, label in [(5, "F1>F2"), (20, "F2>F3")]:
        if ep < max(epochs):
            axes[0].axvline(ep, color="#64748b", linestyle=":", lw=1.2)
            axes[0].text(ep + 0.2, axes[0].get_ylim()[1] * 0.96,
                         label, color="#64748b", fontsize=8)
 
    # F1 Macro
    axes[1].plot(epochs, history["train_f1"], color="#10b981", lw=2, label="Train F1 Macro")
    axes[1].plot(epochs, history["val_f1"],   color="#f43f5e", lw=2, label="Val F1 Macro", linestyle="--")
    best_idx = int(np.argmax(history["val_f1"]))
    best_ep  = epochs[best_idx]
    best_f1  = history["val_f1"][best_idx]
    axes[1].scatter([best_ep], [best_f1], color="#fbbf24", zorder=5, s=90,
                    label=f"Best F1={best_f1:.4f} (e{best_ep})")
    axes[1].set_title("F1 Macro por Epoca — Expert 3 ISIC 2019")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("F1 Macro")
    axes[1].set_ylim(0, 1)
    axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    axes[1].legend(facecolor="#334155", labelcolor="#f1f5f9")
    axes[1].grid(alpha=0.2, color="#475569")
    for ep, label in [(5, "F1>F2"), (20, "F2>F3")]:
        if ep < max(epochs):
            axes[1].axvline(ep, color="#64748b", linestyle=":", lw=1.2)
            axes[1].text(ep + 0.2, 0.03, label, color="#64748b", fontsize=8)
 
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"\n  Grafica guardada en: {save_path}")
 

In [15]:
# Forzar modo alto rendimiento en GPU
torch.cuda.set_per_process_memory_fraction(0.9)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# Verificar que CUDA está activo correctamente
print(f"TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")

TF32 matmul: True
cuDNN benchmark: True


In [16]:
# ─────────────────────────────────────────────
# 9. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────
def main():
    # ── Datos
    print("\n=== Cargando datos ISIC 2019 ===")
    df = build_dataframe(GT_CSV)
    print(f"Total muestras validas: {len(df)}")
    print("\nDistribucion por clase:")
    for i, name in enumerate(CLASS_NAMES):
        n   = (df["label"] == i).sum()
        bar = "#" * (n // 300)
        print(f"  {name:<5} {n:>5}  {bar}")
 
    train_df, val_df = train_test_split(
        df, test_size=0.15, stratify=df["label"], random_state=SEED
    )
    print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,}")
 
    class_weights = compute_class_weights(train_df).to(device)
    print("\nClass weights:")
    for name, w in zip(CLASS_NAMES, class_weights.cpu()):
        print(f"  {name:<5} {w:.4f}")
 
    sampler = build_weighted_sampler(train_df, class_weights.cpu())
 
    # num_workers=0: CRITICO para Windows + Jupyter (evita freeze)
    # persistent_workers eliminado por la misma razon
    train_loader = DataLoader(
        ISICDataset(train_df, train_transform),
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        ISICDataset(val_df, val_transform),
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
 
    # ── Modelo
    print("\n=== Construyendo ConvNeXt-Small ===")
    model = Expert3_ConvNeXtSmall(pretrained=True).to(device)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parametros totales     : {total:,}")
    print(f"  Parametros entrenables : {trainable:,}")
 
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    scaler    = GradScaler("cuda")   # sintaxis nueva: sin FutureWarning
 
    history = {"epoch": [], "train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}
    best_f1      = 0.0
    patience_cnt = 0
 
    # ─────────────────────────────────────────
    # FASE 1 — Solo head (backbone congelado) | Epocas 1-5
    # ─────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  FASE 1 — Head solamente | Backbone CONGELADO | Epocas 1-5")
    print("=" * 65)
    print(HEADER); print(SEP)
 
    model.freeze_backbone()
    now = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [INFO] Params entrenables: {now:,} (solo head)\n")
 
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=5, eta_min=LR_HEAD * 0.1
    )
 
    for epoch in range(1, 6):
        lt, ft = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, NUM_EPOCHS)
        lv, fv, _, _ = validate(model, val_loader, criterion, epoch, NUM_EPOCHS)
        scheduler.step()
        is_best = fv > best_f1
        if is_best:
            best_f1 = fv
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_f1": best_f1, "class_names": CLASS_NAMES}, CHECKPOINT_PATH)
            patience_cnt = 0
        else:
            patience_cnt += 1
        print_epoch_row(epoch, "FASE-1", lt, ft, lv, fv, is_best)
        history["epoch"].append(epoch)
        history["train_loss"].append(lt); history["train_f1"].append(ft)
        history["val_loss"].append(lv);   history["val_f1"].append(fv)
 
    # ─────────────────────────────────────────
    # FASE 2 — Fine-tuning diferencial | Epocas 6-20
    # ─────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  FASE 2 — Fine-tuning ultimos 2 stages | Epocas 6-20")
    print("=" * 65)
    print(HEADER); print(SEP)
 
    model.unfreeze_backbone(last_n_stages=2)
    now = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [INFO] Params entrenables: {now:,}\n")
    patience_cnt = 0
 
    optimizer = torch.optim.AdamW([
        {"params": model.head.parameters(),     "lr": LR_HEAD},
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
 
    for epoch in range(6, 21):
        lt, ft = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, NUM_EPOCHS)
        lv, fv, _, _ = validate(model, val_loader, criterion, epoch, NUM_EPOCHS)
        scheduler.step()
        is_best = fv > best_f1
        if is_best:
            best_f1 = fv
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_f1": best_f1, "class_names": CLASS_NAMES}, CHECKPOINT_PATH)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"\n  [Early Stop] Sin mejora en {PATIENCE} epocas. Pasando a FASE 3.")
                break
        print_epoch_row(epoch, "FASE-2", lt, ft, lv, fv, is_best)
        history["epoch"].append(epoch)
        history["train_loss"].append(lt); history["train_f1"].append(ft)
        history["val_loss"].append(lv);   history["val_f1"].append(fv)
 
    # ─────────────────────────────────────────
    # FASE 3 — Fine-tuning global | Epocas 21-40
    # ─────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  FASE 3 — Fine-tuning global completo | Epocas 21-40")
    print("=" * 65)
    print(HEADER); print(SEP)
 
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.unfreeze_all()
    now = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [INFO] Params entrenables: {now:,} (todos)\n")
    patience_cnt = 0
 
    optimizer = torch.optim.AdamW([
        {"params": model.head.parameters(),     "lr": LR_HEAD     * 0.1},
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE * 0.5},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=20, eta_min=1e-8
    )
 
    final_preds, final_labels = [], []
    for epoch in range(21, NUM_EPOCHS + 1):
        lt, ft = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, NUM_EPOCHS)
        lv, fv, fp, fl = validate(model, val_loader, criterion, epoch, NUM_EPOCHS)
        scheduler.step()
        is_best = fv > best_f1
        if is_best:
            best_f1 = fv
            final_preds, final_labels = fp, fl
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_f1": best_f1, "class_names": CLASS_NAMES}, CHECKPOINT_PATH)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"\n  [Early Stop] Sin mejora en {PATIENCE} epocas consecutivas.")
                break
        print_epoch_row(epoch, "FASE-3", lt, ft, lv, fv, is_best)
        history["epoch"].append(epoch)
        history["train_loss"].append(lt); history["train_f1"].append(ft)
        history["val_loss"].append(lv);   history["val_f1"].append(fv)
 
    # ─────────────────────────────────────────
    # EVALUACION FINAL
    # ─────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  EVALUACION FINAL (mejor checkpoint)")
    print("=" * 65)
 
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    _, final_f1, final_preds, final_labels = validate(
        model, val_loader, criterion, epoch="*", total_epochs="*"
    )
    print(f"\n  Mejor F1 Macro en validacion : {final_f1:.4f}")
    print(f"  Checkpoint guardado en       : {CHECKPOINT_PATH}")
    print(f"  VRAM pico                    : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    print("\nReporte por clase:")
    print(classification_report(final_labels, final_preds,
                                target_names=CLASS_NAMES, zero_division=0))
 
    plot_curves(history, PLOT_PATH)
    print("\n  Entrenamiento completado.")
 
 
if __name__ == "__main__":
    main()


=== Cargando datos ISIC 2019 ===
Total muestras validas: 25331

Distribucion por clase:
  MEL    4522  ###############
  NV    12875  ##########################################
  BCC    3323  ###########
  AK      867  ##
  BKL    2624  ########
  DF      239  
  VASC    253  
  SCC     628  ##

Train: 21,531 | Val: 3,800

Class weights:
  MEL   0.1491
  NV    0.0524
  BCC   0.2029
  AK    0.7775
  BKL   0.2570
  DF    2.8228
  VASC  2.6653
  SCC   1.0731

=== Construyendo ConvNeXt-Small ===


  Parametros totales     : 49,655,144
  Parametros entrenables : 49,655,144

  FASE 1 — Head solamente | Backbone CONGELADO | Epocas 1-5
 Epoca | Fase    |  L_train |  F1_train |    L_val |   F1_val |  VRAM_peak | Best
---------------------------------------------------------------------------------
  [INFO] Params entrenables: 200,456 (solo head)



     1 | FASE-1  |   1.2757 |    0.1699 |   2.8509 |   0.0700 |      0.80G | <<


     2 | FASE-1  |   1.1069 |    0.2188 |   2.7532 |   0.0933 |      0.80G | <<


     3 | FASE-1  |   1.0600 |    0.2382 |   2.6607 |   0.1187 |      0.80G | <<


...

    38 | FASE-3  |   0.4048 |    0.8854 |   1.8332 |   0.6901 |      4.29G | <<


    39 | FASE-3  |   0.4010 |    0.8830 |   1.8279 |   0.6915 |      4.29G | <<


C:\Users\isape\AppData\Local\Temp\ipykernel_14036\756869023.py:189: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=device)


    40 | FASE-3  |   0.3987 |    0.8827 |   1.8285 |   0.6901 |      4.29G |

  EVALUACION FINAL (mejor checkpoint)



  Mejor F1 Macro en validacion : 0.6915
  Checkpoint guardado en       : expert3_convnext_isic2019_best.pth
  VRAM pico                    : 4.29 GB

Reporte por clase:
              precision    recall  f1-score   support

         MEL       0.53      0.82      0.64       678
          NV       0.97      0.58      0.73      1931
         BCC       0.82      0.91      0.86       499
          AK       0.67      0.83      0.74       130
         BKL       0.59      0.87      0.70       394
          DF       0.30      0.89      0.45        36
        VASC       0.47      1.00      0.64        38
         SCC       0.75      0.78      0.76        94

    accuracy                           0.72      3800
   macro avg       0.64      0.83      0.69      3800
weighted avg       0.80      0.72      0.73      3800


  Grafica guardada en: expert3_training_curves.png

  Entrenamiento completado.
